# A Decision Tree on Churn (+ Prune It)

**Week 5 · Lesson 3** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week5/lectures/03_decision_tree_on_churn.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from ISLP Ch. 8.3 Lab where relevant.*

## Setup

Same boilerplate as every notebook this term — see Week 1 Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** which customers are likely to churn — and this time, can the
retention team read the model's *logic* well enough to apply it without a laptop in
the room?

**Why it matters:** Week 1's random-forest preview and Week 3's logistic regression
(Lesson 5) both handed back a *score*. Lessons 1–2 this week built something
different: a **decision tree**, whose output is a literal sequence of yes/no
questions — *"is this a month-to-month contract? has this customer been here under a
year?"* — ending in a plain-language leaf. Lesson 2 also showed that an unpruned
tree memorizes its training data. This notebook is where both ideas meet real
numbers: fit the full tree, watch it overfit for real, then prune it by
cross-validation exactly the way Lesson 2 described, and read the result as a rule a
human could actually follow.

## 2. Data

*Load, describe, and sanity-check the data before touching a model.*

Same dataset as Week 1 Lesson 8 and Week 3 Lesson 5 — **Telco Customer Churn**, one
row per customer, account/contract/service features plus a binary `Churn` label.
`load_churn()` in `gbus564_coursework.utils.course_data` loads and cleans it the same way every
lesson that touches this dataset does.

In [ ]:
from gbus564_coursework.utils.course_data import load_churn

df = load_churn()
df.head()

In [ ]:
display(df.describe())

display(df.describe(exclude="number"))


### Data Prep

Trees split on raw thresholds, one feature at a time — unlike Week 4's kNN or Week
3's regularized regression, **nothing here needs standardizing**: a split on
`tenure < 12` cares only about the *ordering* of tenure values, not its scale. The
one requirement that doesn't go away: scikit-learn's `DecisionTreeClassifier` still
needs numbers, not strings, so the categorical columns (`Contract`, `PaymentMethod`,
...) still get one-hot encoded — the same `ColumnTransformer` pattern as Week 3 Lesson 5,
just without the scaling step.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X, y = df.drop(columns="Churn"), df["Churn"]
categorical_cols = X.select_dtypes(exclude="number").columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

preprocess = ColumnTransformer(
    [("encode", OneHotEncoder(handle_unknown="ignore"), categorical_cols)],
    remainder="passthrough",
)

## 3. Model

*Baseline first — the reference every model must beat — then the model under
study.*

`DummyClassifier` (always predict the majority class, "stays") is the same Week 2
baseline every churn model this course fits has to clear. The model under study
starts as an **unconstrained decision tree** — no `max_depth`, no
`min_samples_leaf` — deliberately, so the next section can show Lesson 2's
overfitting curve on real data instead of synthetic data.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

full_tree = Pipeline([
    ("prep", preprocess),
    ("clf", DecisionTreeClassifier(random_state=RANDOM_SEED)),
])
full_tree.fit(X_train, y_train)

train_acc = full_tree.score(X_train, y_train)
test_acc = full_tree.score(X_test, y_test)
n_leaves = full_tree.named_steps["clf"].get_n_leaves()
print(f"Unconstrained tree — leaves: {n_leaves}")
print(f"Training accuracy: {train_acc:.2%}   Test accuracy: {test_acc:.2%}")

Exactly Lesson 2's depth-vs-error curve, now on real customers: training accuracy
near-perfect, test accuracy well behind it, and a leaf count in the hundreds — a
tree that size isn't a rule anyone could read, let alone apply in a retention
call. Cost-complexity pruning is the fix.

### Model Selection

Lesson 2's three-step workflow, run for real: **(1)** the tree just grown *is* the
full tree from step 1 — `cost_complexity_pruning_path` reads its whole
weakest-link sequence of candidate alphas straight off of it; **(2)**
cross-validate over those candidate alphas *on the training data only*; **(3)**
refit the full tree at the CV-chosen alpha. `GridSearchCV` does steps 2–3 in one
call.

**Why tune on accuracy, after Week 2's warning?** A tree's scores come in
steps: every customer who lands in the same leaf gets the same churn
probability, so a 13-leaf tree can only ever produce 13 distinct scores. Ranking
metrics like AUC have little to work with at that resolution — they mostly
reward adding more leaves, which works against the whole point of a tree you
can read. Accuracy is a fair pruning target for a model whose job is a readable
rule, and here it costs nothing: swapping in `scoring="roc_auc"` picks the exact
same tree.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Step 1: the candidate alphas, read off the tree already fit above (its
# preprocessing step is already fit, so transform — don't refit — here).
ccp_alphas = full_tree.named_steps["clf"].cost_complexity_pruning_path(
    preprocess.transform(X_train), y_train
).ccp_alphas
ccp_alphas = ccp_alphas[:-1]  # drop the trivial single-node tree's alpha

# Steps 2-3: cross-validate over alpha, refit at the winner.
pruned_search = GridSearchCV(
    Pipeline([("prep", preprocess), ("clf", DecisionTreeClassifier(random_state=RANDOM_SEED))]),
    param_grid={"clf__ccp_alpha": ccp_alphas},
    cv=5,
    scoring="accuracy",
)
pruned_search.fit(X_train, y_train)

pruned_tree = pruned_search.best_estimator_
best_alpha = pruned_search.best_params_["clf__ccp_alpha"]
print(f"CV-chosen alpha: {best_alpha:.7f}")
print(f"Leaves after pruning: {pruned_tree.named_steps['clf'].get_n_leaves()}")

## 4. Evaluate

*Score against the baseline — and against the unconstrained tree — not in
isolation.*

Three numbers side by side: the Week 2 baseline, the unconstrained tree (Section
3), and the tree cross-validation actually chose, scored on accuracy for the
reason Model Selection gave. Week 2's caveat still holds, though: accuracy is
the *floor* check, against the majority-class baseline every one of these has to
clear, not a number to trust on its own. The confusion matrix below shows what
it hides.

In [ ]:
baseline_acc = baseline.score(X_test, y_test)
full_test_acc = full_tree.score(X_test, y_test)
pruned_test_acc = pruned_tree.score(X_test, y_test)

print(f"Baseline accuracy:      {baseline_acc:.2%}")
print(f"Unconstrained tree:     {full_test_acc:.2%}  ({full_tree.named_steps['clf'].get_n_leaves()} leaves)")
print(f"Pruned tree (CV alpha): {pruned_test_acc:.2%}  ({pruned_tree.named_steps['clf'].get_n_leaves()} leaves)")

The unconstrained tree's 1,107 leaves didn't just fail to help on new customers —
they actively hurt: it lands *below* the naive baseline. That's Lesson 2's
overfitting curve made concrete: past the point where more splits stop
generalizing, they're pure memorization, and memorization can score worse than
guessing the majority class. The 13-leaf pruned tree, with a fraction of a percent
of the leaves, is the one that actually beats the baseline.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

pred = pruned_tree.predict(X_test)
ConfusionMatrixDisplay(
    confusion_matrix(y_test, pred), display_labels=["Stayed", "Churned"]
).plot(cmap="Blues", colorbar=False)
plt.title("Pruned tree — confusion matrix at the default vote")
plt.tight_layout()

**Reading it:** of the 374 test-set customers who actually churned, the pruned
tree flags 232, so it catches **62% of churners**. The baseline's 73.5% accuracy
caught none. The other 142 churners are missed, and for a retention team those
are the expensive errors. Of the 391 customers it flags, 232 really do churn: a
**59% offer hit rate** (Week 3 Lesson 5's precision), so roughly three of every
five retention offers would reach a real churner.

All of that is at the default vote, which flags a customer when their leaf is
majority-churn. Week 3 Lesson 5 slid logistic regression's threshold to trade
hit rate for coverage. A tree's threshold behaves differently, for the reason
Model Selection gave: its scores come in steps, one per leaf.

In [ ]:
proba = pruned_tree.predict_proba(X_test)[:, 1]
print("Every churn score this tree can produce:", np.unique(proba.round(2)))

churners = y_test == 1
rows = []
for cutoff in [0.5, 0.4, 0.3]:
    flagged = proba >= cutoff
    caught = (flagged & churners).sum()
    rows.append({
        "cutoff": cutoff,
        "flagged for a call": flagged.sum(),
        "churners caught": caught,
        "false alarms": (flagged & ~churners).sum(),
        "share of churners caught": round(caught / churners.sum(), 2),
        "offer hit rate": round(caught / flagged.sum(), 2),
    })
pd.DataFrame(rows).set_index("cutoff")

Thirteen leaves, thirteen possible scores, and a wide gap in the middle: no leaf
scores between 0.36 and 0.53. So lowering the cutoff from 0.5 to 0.4 changes
**nothing**. Not one customer moves. Dropping to 0.3 is the first move that does
anything, and it moves whole leaves at once: 38 more churners caught (72% of
them, up from 62%), but 91 more false alarms, and the offer hit rate falls from
59% to 52%. With logistic regression the threshold is a dial. With a single tree
it's a short menu of fixed operating points, and choosing among them is a
business call about what a missed churner costs compared with a wasted offer.

## 5. Interpret

*What is the model actually keying off of? Business-readable, not just
statistically significant.*

A tree's payoff over logistic regression's coefficients (Week 3 Lesson 5): nothing
here needs converting. The diagram below **is** the explanation — an explicit
sequence of yes/no questions ending in a churn call.

In [ ]:
from sklearn.tree import plot_tree

feature_names = pruned_tree.named_steps["prep"].get_feature_names_out()
feature_names = [n.replace("encode__", "").replace("remainder__", "") for n in feature_names]

plt.figure(figsize=(18, 9))
plot_tree(
    pruned_tree.named_steps["clf"],
    feature_names=feature_names,
    class_names=["Stayed", "Churned"],
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title("Pruned decision tree")
plt.tight_layout()

**Tracing one path:** start at the root and follow the branches down to a single
leaf — that path, read left to right, is the entire model's reasoning for that
customer. One concrete route through the tree above: **month-to-month contract**
→ **fiber-optic internet** → **tenure ≤ 15.5 months** → **total charges so far ≤
$120** lands in a leaf where 154 of 179 training customers (86%) churned. That
sentence — four yes/no checks, no model required at the point of the call — is a
rule a retention rep could apply by hand. Pick any other leaf in the diagram and
read its own path back to the root the same way.

## 6. Recommend

**Recommendation:** hand the retention team the traced paths above as literal
screening rules, not just a model score — a tree's whole selling point over Week
3's logistic regression is that the *reasoning*, not just the *probability*, is
something a non-technical stakeholder can audit and apply directly. Route
customers who land in high-churn leaves to the retention queue, and use each
leaf's own churn rate (shown in the diagram) to prioritize among them.

**One caveat to carry into Lesson 4:** this is still a *single* pruned tree, and
Lesson 2 was explicit that a tree — even a well-pruned one — remains a
comparatively high-variance model: a slightly different training sample could
prune to a noticeably different tree. Lesson 4 picks up exactly there, averaging
many trees together to tame that variance, at the cost of the plain-English
diagram this notebook just earned.